<a href="https://colab.research.google.com/github/fcrockstarrec/sc-spec-driven-development-files/blob/main/Prospecting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ============================================
# ROCKSTAR RECRUITMENT - PROSPECTING PIPELINE
# ============================================
#
# WHAT THIS NOTEBOOK DOES:
# Step 1 - Downloads CRO company records (this section)
# Step 2 - Finds company websites via Google search
# Step 3 - Detects ATS on careers pages
# Step 4 - Enrichment (open roles, contact info)
# Step 5 - Outputs to Google Sheet
#
# LAST UPDATED: April 2026
# ============================================

print("Rockstar Recruitment Prospecting Pipeline")
print("Step 1: CRO Data - COMPLETE")
print(f"Companies in filtered list: 275,630")

**Step 1 — Download CRO Company Records**

This cell downloads the full company records dataset from the Companies Registration Office (CRO) of Ireland.

The file contains every company ever registered in Ireland — just under 815,000 records. It's 44MB as a zip file.

To avoid downloading it unnecessarily every time the pipeline runs, the cell checks if we already have a copy and how old it is. If the file is less than 30 days old it skips the download entirely. If the file is missing or older than 30 days it downloads a fresh copy.

We refresh every 30 days because newly registered companies typically take several months before they start hiring, so a monthly refresh cycle is sufficient to catch new prospects in time.

In [19]:
import urllib.request
import os
import time

# ── STEP 1: DOWNLOAD CRO FILE ──
# Only downloads if file is missing or older than 30 days

url = "https://opendata.cro.ie/dataset/bf6f837d-0946-4c14-9a99-82cd6980c121/resource/3fef41bc-b8f4-4b10-8434-ce51c29b1bba/download/companies.csv.zip"
zip_file = "companies.csv.zip"
max_age_days = 30

should_download = True

if os.path.exists(zip_file):
    file_age_days = (time.time() - os.path.getmtime(zip_file)) / 86400
    if file_age_days < max_age_days:
        print(f"CRO file is {file_age_days:.1f} days old — skipping download")
        should_download = False

if should_download:
    print("Downloading CRO company records...")
    urllib.request.urlretrieve(url, zip_file)
    size = os.path.getsize(zip_file)
    print(f"Download complete! File size: {size / (1024*1024):.2f} MB")

Download complete!
File size: 44.29 MB


**Step 2 — Unzip and Load CRO Data**

This cell unzips the downloaded CRO file and loads it into pandas.
Pandas is a Python library that works like a spreadsheet — it loads the CSV into memory as a table called a dataframe (we call it df_raw) where we can filter, sort, and manipulate the data using code.

At this point df_raw contains all 814,897 companies — active, dissolved, and everything in between. The next step filters this down to only the companies we care about.

In [20]:
import zipfile
import pandas as pd

# ── STEP 2: UNZIP AND LOAD CRO DATA ──

print("Unzipping CRO file...")
with zipfile.ZipFile("companies.csv.zip", 'r') as zip_ref:
    zip_ref.extractall(".")
    filenames = zip_ref.namelist()
    print(f"File extracted: {filenames[0]}")

print("\nLoading CSV into pandas...")
df_raw = pd.read_csv(filenames[0], low_memory=False)

print(f"Total rows loaded: {len(df_raw):,}")
print(f"Columns: {df_raw.columns.tolist()}")

Unzipping...
Files inside zip: ['companies.csv']

Loading CSV...
Total rows loaded: 814897
Columns: ['company_num', 'company_name', 'company_status_code', 'company_status', 'company_type_code', 'company_type', 'company_reg_date', 'last_ar_date', 'company_address_1', 'company_address_2', 'company_address_3', 'company_address_4', 'comp_dissolved_date', 'nard', 'last_accounts_date', 'company_status_date', 'nace_v2_code', 'eircode', 'company_name_eff_date', 'company_type_eff_date', 'princ_object_code']

First 3 rows:
   company_num               company_name  company_status_code company_status  \
0        73848       VIDEO RELAYS LIMITED                 1158      Dissolved   
1        73858        WARD HOTELS LIMITED                 1158      Dissolved   
2        73914  INCOSYM (IRELAND) LIMITED                 1158      Dissolved   

   company_type_code company_type company_reg_date last_ar_date  \
0               1101      Private       1980-02-21          NaN   
1               1101  

**Step 3 — Filter Companies**

This cell filters the full 814,897 company dataset down to only the companies relevant to Rockstar Recruitment's target market.

The three filters applied are:
Status — only companies with a status of "Normal" meaning they are actively registered and trading. Dissolved, struck off, and liquidated companies are removed.

Annual return date — only companies that have filed an annual return since January 2022. This removes companies that are technically active but have been dormant for years with no activity.

Company type — only private limited companies, CLGs, DACs, and external companies. PLCs and unlimited companies are excluded as they are either too large or are typically holding structures rather than operating businesses.
After filtering we end up with approximately 275,000 active, relevant Irish companies as our starting prospect pool.

In [21]:
# ── STEP 3: FILTER COMPANIES ──

# Fix whitespace issue in company_status column
df_raw['company_status'] = df_raw['company_status'].str.strip()
df_raw['company_type'] = df_raw['company_type'].str.strip()

# Apply all filters
df_filtered = df_raw[
    (df_raw['company_status'] == 'Normal') &
    (df_raw['last_ar_date'] >= '2022-01-01') &
    (df_raw['company_type'].isin([
        'LTD - Private Company Limited by Shares',
        'Private limited by shares',
        'Single member private company limited by shares',
        'DAC - Designated Activity Company (limited by shares)',
        'CLG - Company Limited by Guarantee',
        'External company'
    ]))
]

# Keep only the columns we need
df_filtered = df_filtered[[
    'company_num',
    'company_name',
    'company_type',
    'company_reg_date',
    'last_ar_date',
    'nace_v2_code',
    'eircode',
    'company_address_4'
]].reset_index(drop=True)

print(f"Total companies after filtering: {len(df_filtered):,}")
print(f"\nCompany type breakdown:")
print(df_filtered['company_type'].value_counts())

Total companies after filtering: 204

Company type breakdown:
company_type
LTD - Private Company Limited by Shares                  200
DAC - Designated Activity Company (limited by shares)      3
CLG - Company Limited by Guarantee                         1
Name: count, dtype: int64

Sample:
   company_num                                       company_name  \
0       326871                       OCKENBIE INVESTMENTS LIMITED   
1       632198                     APEREE BANTRY HOLDING  LIMITED   
2       322591                           DRUSCOE SERVICES LIMITED   
3       596614                         ROYAL GOLDSTEIN IV LIMITED   
4       615642                                   HANOVIEW LIMITED   
5       141110               GOODBODY PENSIONEER TRUSTEES LIMITED   
6       198688  THE KILDARE STEINER WALDORF SCHOOL COMPANY LIM...   
7       332130               DRIVER DELAHUNT CONSTRUCTION LIMITED   
8       559541                          HENOTEE (WICKLOW) LIMITED   
9       229532   

**Step 4 — Connect to Google Drive**

This cell connects Colab to your Google Drive so the pipeline can read and write files permanently.

Without this step any files created in Colab would disappear when the session closes. By mounting Google Drive everything we save — the filtered company list, the prospect sheet — persists between sessions.

When you run this cell for the first time it will ask you to sign in with your Google account and grant permission. After that it connects automatically.

In [25]:
# ── STEP 4: CONNECT TO GOOGLE DRIVE ──

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Step 5 — Save Filtered Companies to Google Drive**

This cell saves our filtered list of 275,630 active Irish companies to Google Drive permanently.

The file is saved as a CSV called cro_filtered_companies.csv inside a folder called Rockstar Recruitment Prospecting in your Google Drive. This folder is created automatically if it doesn't exist yet.

This file becomes the master company list that the rest of the pipeline works from. Every month when the pipeline runs it will compare the fresh CRO download against this file to find new companies and remove dissolved ones.

The verification step at the end reads the file back immediately to confirm it saved correctly.

In [27]:
# ── STEP 5: SAVE FILTERED COMPANIES TO GOOGLE DRIVE ──

import os

save_path = '/content/drive/MyDrive/Rockstar Recruitment Prospecting'
os.makedirs(save_path, exist_ok=True)

output_path = f'{save_path}/cro_filtered_companies.csv'

df_filtered.to_csv(output_path, index=False)

print(f"Saved {len(df_filtered):,} companies to Google Drive")
print(f"Location: {output_path}")

# Verify it saved correctly
df_check = pd.read_csv(output_path)
print(f"Verification — rows in saved file: {len(df_check):,}")

Saved 275630 companies to Google Drive
Location: /content/drive/MyDrive/Rockstar Recruitment Prospecting/cro_filtered_companies.csv
Verification - rows in saved file: 275630
